# Transformers, what can they do?

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.  
Install pytorch

In [ ]:
%pip install datasets evaluate "transformers[sentencepiece]"
%pip install torch
%pip install -U huggingface_hub
%pip install ipywidgets

In [5]:
import torch, transformers
from transformers import pipeline

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

PyTorch: 2.12.1
Transformers: 5.12.1


In [6]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(device)   

mps


In [3]:
## pipeline() will be extremely slow without hf_token
import os
# os.environ["HF_TOKEN"] = "<REDACTED_HF_TOKEN>"

Some old NLP tasks using conventional models

In [ ]:
classifier = pipeline("sentiment-analysis")
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

In [ ]:
classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

In [ ]:
unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

Since hugging-face transformers V5, it is suggested that all text-generation tasks should use LLMs

In [2]:
# download a model from the Hugging Face Hub, to avoid network issues.
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct",
    cache_dir="./model_cache"
)

print("Downloaded to:", model_path)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Downloaded to: ./model_cache/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775


In [8]:
import time

dtype = torch.float16 if device == "mps" else torch.float32

print("Using:", device)
print("Loading model...")
start = time.time()

qa = pipeline(
    "text-generation",
    model=model_path,
    device=device,
    dtype=dtype,
)

print(f"Loaded in {time.time() - start:.1f} seconds")

Using: mps
Loading model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded in 0.7 seconds


In [9]:

context = """
The Hugging Face Transformers library provides pretrained models for
natural-language processing, computer vision, and audio. The pipeline
function offers a simple interface for running model inference.
"""

question = "What does the pipeline function provide?"

messages = [
    {
        "role": "system",
        "content": (
            "Answer the question using only the supplied context. "
            "If the answer is not present, say: I don't know."
        ),
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestion: {question}",
    },
]

result = qa(
    messages,
    max_new_tokens=50,
    do_sample=False,
)

answer = result[0]["generated_text"][-1]["content"]
print(answer)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The pipeline function provides a simple interface for running model inference.


In [20]:

context = """
My name is Sylvain and I work at Hugging Face in Brooklyn."""

questions = [
    "what is my name?",
    "where am I working?",
    "which company do I work for?"
]

messages = [
    {
        "role": "system",
        "content": (
            "Answer the questions one by one,using only the supplied context. "
            "If the answer is not present, say: I don't know."
        ),
    },
    {
        "role": "user",
        "content": f"Context:\n{context}\n\nQuestions: {questions}",
    },
]

result = qa(
    messages,
    max_new_tokens=50,
    do_sample=False,
)

answer = result[0]["generated_text"][-1]["content"]
print(answer)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


My name is Sylvain.
I work at Hugging Face in Brooklyn.
I work for Hugging Face.


In [ ]:
from transformers import pipeline

image_classifier = pipeline(
    task="image-classification", model="google/vit-base-patch16-224"
)
result = image_classifier(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
print(result)

In [ ]:
from transformers import pipeline

transcriber = pipeline(
    task="automatic-speech-recognition", model="openai/whisper-large-v3"
)
result = transcriber(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
)
print(result)